# 多星座 FSPL 数据生成

第一版：A 为 victim 地面站，连接一颗 A 星；B 为 aggressor 地面站，连接一颗 B 星。B 当前活动星的同频下行在 A 处形成干扰。仅 FSPL、固定且相同的发射功率/增益、固定接收增益和热噪声。**位置来自轨道 CSV，通信行为是合成实验。**

只从 CSV 读取轨道，不联网、不重复传播轨道。输入与场景边界、换星评分说明见同目录 `README.md`。

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "generate data"]
             if (p / "dataset.py").is_file() and (p / "simulation.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside WorldNTN.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from config import SimulationConfig, GroundStation, RadioConfig, TriggerConfig, RewardConfig, ground_east_of, constellation_name
from orbit_io import discover_csvs
from dataset import run_dataset, link_budget_table, TrainingView
# Install dependencies in this kernel if needed:
# %pip install -r "{ROOT / 'requirements.txt'}"
SATELLITE_ROOT = ROOT.parent / "generate satellite"
print("Data generator:", ROOT)


## 选择星座与轨道 CSV

`amazon` 自动映射到 `kuiper`。每方可以指定一个真实 CSV，或一个真实 CSV 加一个虚拟补全 CSV；不要同时加入两种替代虚拟方案。`None` 自动选择该星座最新的 **真实 all_states** CSV，不会自动叠加虚拟数据。

原生成器默认只有 visible CSV。请在 `generate satellite/scenario.json` 设置 `write_all_states=true` 后重跑所需 tracker。若研究秒级切换，建议上游同时把 `interval_sec` 改成 1–5 秒。这里不把 120 秒轨道样本伪装成秒级数据。

In [ ]:
victim = "starlink"
aggressor = "amazon"

# Explicit examples: [Path("..._all_states.csv"), Path("...synthetic..._all_states.csv")]
VICTIM_CSVS = None
AGGRESSOR_CSVS = None
available = discover_csvs(SATELLITE_ROOT)
display(available)

def latest_real_csv(constellation):
    matching = available[(available.constellation == constellation_name(constellation))
                         & (available.coverage == "all_states")
                         & (available.source_type == "real_gp")
                         & (available.status == "complete")]
    return [Path(matching.iloc[-1].path)] if len(matching) else []

selected_victim = [Path(p) for p in VICTIM_CSVS] if VICTIM_CSVS is not None else latest_real_csv(victim)
selected_aggressor = [Path(p) for p in AGGRESSOR_CSVS] if AGGRESSOR_CSVS is not None else latest_real_csv(aggressor)
print("Victim inputs:", selected_victim)
print("Aggressor inputs:", selected_aggressor)


## 射频、地面站、策略和触发规则

推荐基准发射功率为 **10 W/活动卫星/250 MHz 信道**，不是功率谱密度，也不是全星总功率。发射、接收增益暂取各 30 dBi，双方相同。下面直接显示链路预算供调整。

默认 A 初始和 B 均使用 `longest_remaining`；A 换星使用 `random`，可改 `longest_remaining` 或 `max_snr` 做对照。后两者使用冻结轨道推算的 SNR，不偷看未来干扰。

分数为：SINR < 0 dB 得 5 分；[0,3) 得 3 分；[3,6) 得 2 分；≥6 得 1 分。最近 5 个采样点累计 ≥10 在下一时刻执行切换；成功切换后清空旧分数。门限是研究配置，需看分布再校准。

In [ ]:
ground_a = GroundStation(lat_deg=39.95697, lon_deg=-105.16033, height_m=1660.0)
ground_b = ground_east_of(ground_a, distance_m=1000.0)
radio = RadioConfig(
    frequency_hz=20e9, bandwidth_hz=250e6, tx_power_w=10.0,
    tx_gain_dbi=30.0, rx_gain_dbi=30.0,
    temperature_k=290.0, noise_figure_db=6.0, interference_enabled=True,
)
trigger = TriggerConfig(
    sinr_min_db=0.0, middle_start_db=3.0, highest_start_db=6.0,
    window_samples=5, trigger_points=10, bin_points=(5, 3, 2, 1),
    force_on_geometry_loss=False,
)
reward = RewardConfig(
    time_scale_s=60.0, dwell_weight=1.0, quality_weight=1.0,
    outage_weight=2.0, handover_cost=0.05, success_score_threshold=2.0,
)
cfg = SimulationConfig(
    victim=victim, aggressor=aggressor, ground_a=ground_a, ground_b=ground_b,
    radio=radio, trigger=trigger, reward=reward,
    min_snr_db=0.0, min_elevation_deg=25.0,
    aggressor_policy="longest_remaining",
    victim_initial_policy="longest_remaining", victim_handover_policy="random",
    random_seed=20261005, start_time_utc=None, max_steps=600,
    history_samples=10, prediction_samples=1, window_stride=1,
    split="train", scenario_group_id=None,
    allow_partial_catalog=False, allow_unverified_csv=False, assume_naive_utc=False,
).validate()
display(link_budget_table(radio))
print("B site:", ground_b)


## 生成数据

原始轨道 CSV 不修改。每次生成独立 UTC 时间目录；保存输入哈希、参数、种子、代码版本和质量统计。

`public/observations.csv` 只记录 A 已连接方向的 SINR/SNR/INR 和入射方向；`public/geometry.npz` 记录所有输入 A/B 卫星的位置、速度和有效性 mask。B 的实际连接、私有地面站和策略真值放入 `private/`，不作为训练特征。

In [ ]:
RUN_GENERATION = True
run_directory = None
if not selected_victim or not selected_aggressor:
    print("No complete pair selected. Export all_states CSVs upstream or set VICTIM_CSVS / AGGRESSOR_CSVS, then rerun.")
elif RUN_GENERATION:
    run_directory = run_dataset(selected_victim, selected_aggressor, cfg, ROOT / "outputs")
    print("Dataset:", run_directory)


## 检查触发、动作和换星得分

区分“切换执行成功”和“随后链路质量得分合格”。最后一段连接的 `right_censored=True`，不能把受 CSV 终止影响的短驻留判成失败。没有备用星时保留当前状态并记 `blocked_no_candidate`，不会假装换星成功。

In [ ]:
if run_directory is not None:
    display(json.loads((run_directory / "quality_report.json").read_text()))
    observations = pd.read_csv(run_directory / "public" / "observations.csv")
    actions = pd.read_csv(run_directory / "public" / "actions.csv")
    segments = pd.read_csv(run_directory / "labels" / "link_segments.csv")
    display(observations.head(12))
    display(actions[actions.action != "hold"].head(12))
    display(segments.head(12))
    display(observations.quality_points.value_counts(normalize=True).rename("fraction"))


## 训练视图：历史状态 + 本时刻动作 → 下一时刻观测

默认一步预测；先验证图里的动作条件预测，再做多步 rollout。`TrainingView` 不读取 `private/` 或未来换星得分。NaN 会转换为 0，同时给出 mask；不应把填充值当成 0 dB 的真实测量。

不同随机种子或策略的同一基础轨道/站点归入同一 `scenario_group_id`。不要随机打散重叠滑动窗口做 train/test。`link_segments.csv` 的 score 是动作结果标签，不能加回动作发生前的特征。

In [ ]:
if run_directory is not None:
    training = TrainingView(run_directory)
    if len(training.windows):
        sample = training.sample(0)
        print("Feature names:", sample["inputs"]["feature_names"])
        print("History shape:", sample["inputs"]["history"].shape)
        print("A orbit history shape:", sample["inputs"]["a_position_m"].shape)
        print("B orbit history shape:", sample["inputs"]["b_position_m"].shape)
        print("Action:", sample["inputs"]["action"])
        print("Target:", sample["labels"])
    else:
        print("Too few samples for the configured history and prediction windows.")
    training.close()


## 重复采集与对照

使用 `replace(cfg, random_seed=...)` 重复随机探索；改 `victim_handover_policy` 比较最长连接和最大 SNR；设 `radio=replace(radio, interference_enabled=False)` 做无干扰对照。使用相同输入、地面站、时间轴及分组，B 的轨迹和连接规则保持相同。

恒定增益下，换 A 星不会因接收方向变化而抑制 B 干扰；它只改变期望信号、驻留和动作时机。因此这版适合验证数据管线、时序与策略，后续研究方向避干扰时需要加入角度相关增益。